In [3]:
import ollama

In [2]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }

In [5]:
# TOOL 1:CHECKING DATABASE
def check_inventory(product_name):
    product_name = product_name.lower()
    
    if product_name in inventory_db:
        return inventory_db[product_name]
    
    return {"stock": 0, "base_price": None}





# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.30)  # max 30%
    final_price = base_price * (1 - discount)
    return round(final_price, 2)

In [6]:
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [8]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [22]:
message = [
    {"role":"user", "content":"I want to buy a iPhone, can you check the stock"}
]

response = ollama.chat(
    model = "qwen3.5:0.8b",
    messages = message,
    tools = tools
)

In [24]:
print(response["message"])

role='assistant' content='' thinking='The user is asking me to check the stock for a iPhone. I have a tool called `check_inventory` that can help with this - it\'s for checking stock and price for a specific product.\n\nHowever, I notice that the user has specified "iPhone" as the product name I should look up. Let me use this tool.' images=None tool_name=None tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'iPhone'}))]


In [31]:
tool_calls  = response["message"].get("tool_calls")

if tool_calls:
    for tool in tool_calls:
        tool_name = tool['function']['name']
        tool_args = tool['function']['arguments']

        function_to_call = available_functions[tool_name]

        result = function_to_call(**tool_args)

        message.append(response["message"])

        message.append(
            {
                'role':'tool',
                'content':str(result)
            }
        )



final_response = ollama.chat(
    model="qwen3.5:0.8b",
    messages = message
)       

In [34]:
print(final_response["message"]["content"])


The inventory check indicates that there are no available units of the iPhone currently available in stock. However, you could consider alternatives such as:

1. **Official Retailers**: Visit the Apple Store or authorized resellers for best pricing and official warranty.
2. **Coupon Sites**: Check sites like **Apple's official site** or third-party coupons like **Carrd**, **Groupon**, or **Apple Deals** to find current discounts.
3. **Third-Party Resellers**: Sometimes third-party retailers offer faster shipping or better deals.
4. **Buying in Bulk**: Check if there are bulk buying options on the Apple website.

For the best pricing and availability, I recommend contacting a local retailer or checking official Apple sources first before making a purchase. Let me know if you'd like help with any further purchasing options!
